# checkpoint

> Training that outlives a session: checkpoints that hold what trains and where the fit stood, a fit that stops before a deadline and says so, and a learner that picks up from a checkpoint, skipping the fits an earlier session finished and resuming the one it stopped in.

In [ ]:
#| default_exp checkpoint

In [ ]:
#| export
import datetime, json, os, re, shutil, signal, sys, threading, time, warnings
from pathlib import Path

import torch
from fastcore.basics import patch
from safetensors.torch import load_file, save_file
from transformers import TrainerCallback

from sysone.learner import DecisionTrainer, Learner
from sysone.record import jsonable

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import logging, tempfile
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

In [ ]:
#| hide
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.record import training_record

Colab and Kaggle give a session a few hours, then take the machine back. A training that needs longer has to stop in time, keep what it has done, and carry on in the next session as if it had never stopped. This notebook gives the learner three things for that:

- **checkpoints** (`learn.checkpoints(path)`): at the end of every epoch, every `every` minutes, and whenever a fit stops early, the weights that train, the optimizer, the scheduler, the random generators and where the fit stood, beside the learner's own state (its temperatures, its history, its fits so far);
- **a deadline** (`learn.stop_at(deadline, wrap_up)`): a fit stops `wrap_up` minutes before the deadline, at the end of a step, saves a checkpoint and raises `TrainingPaused`, so nothing after it runs on a half-trained model; a SIGTERM does the same;
- **resuming** (`learn.resume_from(path)`): the same code, run again, skips the fits the checkpoint finished, with their weights loaded, and resumes the one it stopped in, at the step it stopped at.

A [sysone job](40_cloud.ipynb#sessions) sets all three up from its environment, so a training script needs no change to run over several sessions:

```python
learn = decision_learner(data, "answerdotai/ModernBERT-base", train="lora")
learn.fit_one_cycle(10, 1e-3)          # in a job: checkpoints, a deadline, and a resume, from the environment
learn.export(job_output())
```

The Trainer does most of it: `transformers.Trainer` saves and restores the optimizer, the scheduler, the random states and the step, and skips the batches an interrupted epoch has seen. sysone decides when to save and when to stop, saves only the weights that train, and keeps the learner's state beside the Trainer's.

## What a checkpoint holds

A Trainer checkpoint normally holds the whole model. Here that would copy a frozen encoder of hundreds of megabytes, or gigabytes, into every checkpoint, though it is the same as the one the job loads from the Hub. A checkpoint keeps `trained_state`: the whole head (its weights and its standardisation), and the encoder's weights that have trained in any fit so far, which `_run` notes as `model.trained_names`. Under LoRA that is the adapter; after a full fine-tune, the whole encoder. `DecisionTrainer.save_model` writes it as `model.safetensors`, and `_load_from_checkpoint` loads it back, with every tensor required to find its place.

In [ ]:
#| export
class TrainingPaused(RuntimeError):
    "A fit stopped before its end, to keep a deadline or on SIGTERM, after saving a checkpoint to resume from"

STATE, WEIGHTS = "sysone_state.json", "model.safetensors"

def trained_state(model) -> dict:
    "The tensors of what has trained: the whole head, and the encoder weights that have trained in a fit (`model.trained_names`)"
    names = getattr(model, "trained_names", set())
    return {k: v.detach().to("cpu").clone() for k, v in model.state_dict().items() if k.startswith("head.") or k in names}

def load_trained(model, folder):
    "Load a checkpoint's weights into `model`; each of them must find its place"
    missing, unexpected = model.load_state_dict(load_file(str(Path(folder) / WEIGHTS)), strict=False)
    if unexpected: raise ValueError(f"{folder}: the checkpoint has weights this model lacks ({unexpected[:3]}): resume with the model it was saved from")
    return model

@patch
def save_model(self:DecisionTrainer, output_dir:str|None=None, _internal_call:bool=False):
    "Write the weights that train (`trained_state`) as `model.safetensors`, from the main process: a checkpoint never copies a frozen encoder"
    output_dir = output_dir or self.args.output_dir
    if self.args.should_save:
        os.makedirs(output_dir, exist_ok=True)
        save_file(trained_state(self.accelerator.unwrap_model(self.model)), os.path.join(output_dir, WEIGHTS), metadata={"format": "pt"})

@patch
def _load_from_checkpoint(self:DecisionTrainer, resume_from_checkpoint:str, model=None):
    load_trained(model if model is not None else self.model, resume_from_checkpoint)

In [ ]:
FIX = "fixtures/tiny-modernbert"
def tiny(train="lora", seed=0, **kw):
    "The fixtures' learner, its random weights drawn from `seed`"
    torch.manual_seed(seed)
    data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25, calib=0.2)
    return decision_learner(data, FIX, train=train, preset="cpu", logging_steps=1, disable_tqdm=True, **kw)

lh, ll = tiny("head"), tiny("lora")
lh.fit(1, lr=1e-3); ll.fit(1, lr=1e-3)
sh, sl = trained_state(lh.model), trained_state(ll.model)
test_eq(all(k.startswith("head.") for k in sh), True)                                          # a frozen encoder stays out
test_eq(all(k.startswith("head.") or "lora_" in k for k in sl), True)                         # under LoRA, the adapter joins the head
assert any("lora_" in k for k in sl)
print(f"head only: {len(sh)} tensors; LoRA: {len(sl)}; the whole model: {len(ll.model.state_dict())}")

{'loss': '1.092', 'grad_norm': '1.242', 'learning_rate': '0', 'epoch': '0.125'}
{'loss': '0.9912', 'grad_norm': '1.023', 'learning_rate': '0.001', 'epoch': '0.25'}
{'loss': '0.6847', 'grad_norm': '0.8457', 'learning_rate': '0.001', 'epoch': '0.375'}
{'loss': '1.446', 'grad_norm': '0.9502', 'learning_rate': '0.001', 'epoch': '0.5'}
{'loss': '1.073', 'grad_norm': '0.7543', 'learning_rate': '0.001', 'epoch': '0.625'}
{'loss': '1.415', 'grad_norm': '1.207', 'learning_rate': '0.001', 'epoch': '0.75'}
{'loss': '1.137', 'grad_norm': '1.131', 'learning_rate': '0.001', 'epoch': '0.875'}
{'loss': '0.846', 'grad_norm': '1.289', 'learning_rate': '0.001', 'epoch': '1'}
{'eval_loss': '1.191', 'eval_accuracy': '0.36', 'eval_soft_accuracy': '0.3767', 'eval_brier': '0.26', 'eval_kl': '0.4434', 'eval_tv': '0.366', 'eval_ece': '0.03599', 'eval_score_mae': '0.838', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0.4286', 'eval_accuracy_score': '0.2', 'eval_accuracy_noul': '0.5', 'eval_n_u

## Checkpointer

`Checkpointer` is a Trainer callback, one per learner. It acts only within a fit (`lr_find`'s range test is none: nothing in it is saved or stopped). At the end of each step it times the step and asks two questions. Has `every` minutes passed since the last checkpoint? Then save. Would one more step, and the checkpoint after it, run into the wrap-up before the deadline, or has a SIGTERM come? Then save and stop. On several GPUs every process decides alike: their answers are combined, so none stops a step before the others. At the end of an epoch it always saves, so a finished fit leaves its final state; a fit that is stopping skips the epoch's evaluation.

When the Trainer has written a checkpoint (`on_save`), the checkpointer removes the older ones, writes the learner's state beside it in `sysone_state.json`, uploads the folder to the Hub when asked (`push`, a private dataset repo; a Colab job needs this, since the VM's disk goes with the VM), and prints a progress line in a job. The state says which fit the checkpoint belongs to, how many fits have finished, whether the run paused, and holds what the learner keeps outside the model: its temperatures and act thresholds, which weights have trained, the losses and metrics so far, and the [fit log](05a_record.ipynb) that the training record reads.

The folder holds one Trainer checkpoint at a time, in a subfolder per fit:

```
checkpoints/
  sysone_state.json
  fit-1/checkpoint-48/      model.safetensors  optimizer.pt  scheduler.pt  rng_state.pth  trainer_state.json
```

In [ ]:
#| export
def _say(event, **kw):
    "A progress line in a job (see track), to the process's own stdout"
    if os.environ.get("SYSONE_PROGRESS"):
        out = sys.__stdout__ if os.environ.get("SYSONE_JOB") else sys.stdout
        print("SYSONE " + json.dumps(jsonable({"t": round(time.time(), 1), "event": event} | kw)), file=out, flush=True)

def _job_id():
    try: return json.loads(os.environ.get("SYSONE_JOB") or "{}").get("job")
    except json.JSONDecodeError: return None

class Checkpointer(TrainerCallback):
    "Save resumable checkpoints in `path` (every `every` minutes and at every epoch's end), stop a fit `wrap_up` minutes before `deadline`, and resume from a checkpoint"
    def __init__(self, learner, path, every:float|None=30, push:str|None=None, deadline:float|None=None, wrap_up:float=10):
        self.learner, self.path, self.every, self.push, self.deadline, self.wrap_up = learner, Path(path), every, push, deadline, wrap_up
        self.resume, self.ckpt, self.fits_done, self.fit_args = None, None, 0, None
        self.stopping, self.reason, self.resumed_step, self.sigterm, self.active = False, None, None, False, False
        self._save_s, self._step_s, self._old = 0.0, None, None

    def fit_dir(self, learn) -> Path: return self.path / f"fit-{learn.fits_started}"

    def on_train_begin(self, args, state, control, **kw):
        self._t_step = self._t_save = time.time()
        if self.active and threading.current_thread() is threading.main_thread():
            try: self._old = signal.signal(signal.SIGTERM, self._on_sigterm)
            except ValueError: self._old = None

    def _on_sigterm(self, signum, frame): self.sigterm = True

    def _agree(self, args, *flags) -> tuple:
        "Each flag, true when it is true on any process"
        dist = torch.distributed
        if not (dist.is_available() and dist.is_initialized()) or dist.get_world_size() < 2: return flags
        t = torch.tensor([float(f) for f in flags], device=args.device)
        dist.all_reduce(t, op=dist.ReduceOp.MAX)
        return tuple(bool(x) for x in t.tolist())

    def on_step_end(self, args, state, control, **kw):
        if not self.active: return                             # lr_find's range test is no fit: nothing to save, nothing to stop
        now = time.time()
        dt, self._t_step = now - self._t_step, now
        self._step_s = dt if self._step_s is None else 0.8 * self._step_s + 0.2 * dt
        late = self.deadline is not None and now + self._step_s + self._save_s > self.deadline - 60 * self.wrap_up
        due = self.every is not None and now - self._t_save >= 60 * self.every
        stop, save = self._agree(args, self.sigterm or late, due)
        if stop:
            self.stopping, self.reason = True, "sigterm" if self.sigterm else "deadline"
            control.should_save = control.should_training_stop = True
        elif save: control.should_save = True
        if control.should_save: self._t_save_start = now
        return control

    def on_epoch_end(self, args, state, control, **kw):
        if not self.active: return
        if self.stopping: control.should_evaluate = False      # saved at the step it stopped at; a partial epoch is not evaluated
        else: control.should_save, self._t_save_start = True, time.time()
        return control

    def on_evaluate(self, args, state, control, **kw):
        if self.active: self._t_save_start = time.time()       # the Trainer evaluates, then saves: time the save alone

    def on_save(self, args, state, control, **kw):
        if not (self.active and state.is_world_process_zero): return
        folder = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        self.ckpt = {"fit": self.learner.fits_started, "trainer": str(folder.relative_to(self.path)), "step": state.global_step,
                     "max_steps": state.max_steps, "epoch": round(state.epoch or 0, 4), "epochs": state.num_train_epochs}
        for old in self.path.glob("fit-*/checkpoint-*"):
            if old != folder: shutil.rmtree(old, ignore_errors=True)
        for d in self.path.glob("fit-*"):
            if d.is_dir() and not any(d.iterdir()): d.rmdir()
        self.write_state(paused=self.stopping)
        pushed = self._push(f"step {state.global_step}") if self.push else None
        self._t_save = time.time()
        self._save_s = self._t_save - getattr(self, "_t_save_start", self._t_save)
        _say("checkpoint", step=state.global_step, max_steps=state.max_steps, epoch=round(state.epoch or 0, 3),
             seconds=round(self._save_s, 1), pushed=pushed)

    def on_train_end(self, args, state, control, **kw): self.release()

    def release(self):
        "Give SIGTERM its handler back and stop acting on steps (also when a fit fails)"
        self.active = False
        if self._old is not None: signal.signal(signal.SIGTERM, self._old); self._old = None

`write_state` writes the state as a new file and renames it over the old one, so a session cut off mid-write leaves the previous state whole. A push is one Hub commit: the repo holds the latest checkpoint, and its history is squashed after each push, so old checkpoints do not pile up in your storage. A push that fails is reported and training goes on; the checkpoint is still on the machine's disk.

In [ ]:
#| export
@patch
def write_state(self:Checkpointer, paused:bool=False):
    "Write the learner's state beside the latest checkpoint, atomically"
    if self.ckpt is None: return
    learn, m = self.learner, self.learner.model
    st = {"version": 1, **self.ckpt, "fits_done": self.fits_done, "paused": paused, "reason": self.reason if paused else None,
          "saved": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), "job": _job_id(), "fit_args": self.fit_args,
          "learner": {"temperatures": learn.temperatures, "act_threshold": learn.act_threshold, "act_score": learn.act_score,
                      "encoder_trained": bool(getattr(m, "encoder_trained", False)), "base_trained": bool(getattr(m, "base_trained", False)),
                      "trained_names": sorted(getattr(m, "trained_names", set()))},
          "recorder": {"losses": learn.recorder.losses, "lrs": learn.recorder.lrs, "metrics": learn.recorder.metrics},
          "fits": learn.fit_log.fits}
    self.path.mkdir(parents=True, exist_ok=True)
    tmp = self.path / (STATE + ".tmp")
    tmp.write_text(json.dumps(jsonable(st), indent=1)); os.replace(tmp, self.path / STATE)

@patch
def _push(self:Checkpointer, message:str) -> bool:
    "Upload the checkpoint folder to the private Hub dataset repo `push` (`owner/name` or `owner/name/folder`), replacing what is there"
    try:
        from huggingface_hub import HfApi
        parts = self.push.removeprefix("hf:").split("/")
        repo, sub = "/".join(parts[:2]), "/".join(parts[2:]) or None
        api = HfApi()
        api.create_repo(repo, repo_type="dataset", private=True, exist_ok=True)
        api.upload_folder(repo_id=repo, repo_type="dataset", folder_path=str(self.path), path_in_repo=sub, delete_patterns="*",
                          commit_message=f"sysone checkpoint: {message}")
        try: api.super_squash_history(repo_id=repo, repo_type="dataset")
        except Exception: pass
        return True
    except Exception as e:
        warnings.warn(f"the checkpoint was not pushed to {self.push}: {e}")
        return False

## Before and after a fit

`_run` asks the checkpointer before each fit (`begin`) and tells it after (`end`). The fits are counted from one, in the order the code starts them, which is the same in every session that runs the same code. With a checkpoint to resume from:

- a fit **the checkpoint finished** is skipped. When the checkpoint holds that fit's final state, its weights are loaded, with the learner's state: temperatures, act thresholds, history, fit log. When a later fit is the checkpoint's, the weights wait for it, since the code in between may change the model (add an adapter, say);
- the fit **the checkpoint stopped in** resumes: its weights and the learner's state are loaded, and the Trainer picks up its optimizer, scheduler, random states and step from the checkpoint and skips the batches the epoch has seen. A fit called with other arguments than it was saved with gets a warning, since its schedule would not line up;
- **later fits** run as usual.

After a fit, `end` notes on the fit log's entry which job ran it, the step it resumed from and why it stopped, and writes the state again (and pushes it, with `push`: a job that resumes from the Hub reads this last state). A fit that stopped writes the state as paused and raises `TrainingPaused`; one that ended counts as done, and the state is written again after the calibration, with the temperatures.

In [ ]:
#| export
@patch
def begin(self:Checkpointer, learn, fit_args:dict):
    "Before a fit: 'skip' (an earlier session finished it), the Trainer checkpoint to resume it from, or None (start it)"
    i, st = learn.fits_started, self.resume
    self.stopping, self.reason, self.resumed_step, self.sigterm, self.active = False, None, None, False, True
    self.fit_args, self._step_s = jsonable(fit_args), None
    if st is None: return None
    if st.get("trainer") is None and st["fits_done"]: self.active = False; raise ValueError(f"{self.path}: the state says {st['fits_done']} fit(s) finished but holds no checkpoint")
    if i <= st["fits_done"]:                                       # finished in an earlier session
        if i == st["fit"]: self._load(learn)
        self.fits_done, self.active = i, False
        return "skip"
    if i == st["fit"]:                                             # the fit the checkpoint stopped in
        self._load(learn)
        if st.get("fit_args") is not None and st["fit_args"] != self.fit_args:
            warnings.warn(f"fit {i} resumes with other arguments than it was saved with: {st['fit_args']} then, {self.fit_args} now")
        self.resumed_step = st["step"]
        return str(self.path / st["trainer"])
    return None

@patch
def _load(self:Checkpointer, learn):
    "The checkpoint's weights into the model, and the learner's state into the learner"
    st, m = self.resume, learn.model
    load_trained(m, self.path / st["trainer"])
    L = st["learner"]
    learn.temperatures, learn.act_threshold, learn.act_score = dict(L["temperatures"] or {}), L["act_threshold"], L["act_score"]
    m.encoder_trained, m.base_trained = L["encoder_trained"], L["base_trained"]
    m.trained_names = getattr(m, "trained_names", set()) | set(L["trained_names"])
    r = st["recorder"]
    learn.recorder.losses, learn.recorder.lrs, learn.recorder.metrics = list(r["losses"]), list(r["lrs"]), list(r["metrics"])
    learn.fit_log.fits = list(st["fits"])
    self.ckpt = {k: st[k] for k in ("fit", "trainer", "step", "max_steps", "epoch", "epochs")}

@patch
def end(self:Checkpointer, learn):
    "After a fit: note it in the fit log and the state; a fit that stopped raises `TrainingPaused`"
    i = learn.fits_started
    if learn.fit_log.fits:
        learn.fit_log.fits[-1] |= {k: v for k, v in (("job", _job_id()), ("resumed_from_step", self.resumed_step), ("stopped", self.reason)) if v is not None}
    main = learn.trainer.is_world_process_zero()
    if self.stopping:
        self.write_state(paused=True)
        if self.push and main: self._push("paused")            # the state as it ends the session, with this fit's entry
        c = self.ckpt or {}
        if main: _say("paused", reason=self.reason, fit=i, step=c.get("step"), max_steps=c.get("max_steps"), epoch=c.get("epoch"))
        raise TrainingPaused(f"fit {i} stopped ({self.reason}) at step {c.get('step')} of {c.get('max_steps')}, epoch {c.get('epoch')} of "
                             f"{c.get('epochs')}; its checkpoint is in {self.path}" + ("; job.resume() picks up from it" if _job_id() else ""))
    self.fits_done = i
    self.write_state(paused=self.sigterm)
    if self.push and main: self._push(f"fit {i} done")
    if self.sigterm: raise TrainingPaused(f"fit {i} ended, then a SIGTERM came; its checkpoint is in {self.path}")

## On the learner

`learn.checkpoints(path)` turns checkpoints on, `learn.stop_at(deadline)` sets a deadline (a Unix time; it turns checkpoints on in the learner's own folder if they are not on), and `learn.resume_from(path)` reads a checkpoint to pick up from. Resuming writes its new checkpoints to the same folder, or copies the checkpoint into the learner's checkpoint folder when that is another one, so the folder always holds the latest. A folder with no state in it is nothing to resume: the learner starts afresh.

In [ ]:
#| export
@patch
def _checkpointer(self:Learner, path=None) -> Checkpointer:
    if self.checkpointer is None:
        self.checkpointer = Checkpointer(self, path or self.path / "checkpoints")
        self.trainer.add_callback(self.checkpointer)
    elif path is not None: self.checkpointer.path = Path(path)
    return self.checkpointer

@patch
def checkpoints(self:Learner, path=None, every:float|None=30, push:str|None=None) -> Learner:
    "Save a resumable checkpoint in `path` at every epoch's end, every `every` minutes, and when a fit stops early; `push` uploads each to a private Hub dataset repo"
    ck = self._checkpointer(path)
    ck.every, ck.push = every, push
    return self

@patch
def stop_at(self:Learner, deadline:float, wrap_up:float=10) -> Learner:
    "Stop a fit `wrap_up` minutes before `deadline` (a Unix time), with a checkpoint, and raise `TrainingPaused`"
    ck = self._checkpointer()
    ck.deadline, ck.wrap_up = float(deadline), float(wrap_up)
    return self

@patch
def resume_from(self:Learner, path) -> Learner:
    "Pick up from the checkpoint in `path`: the fits it finished are skipped, their weights loaded, and the fit it stopped in resumes"
    path = Path(path)
    if not (path / STATE).exists(): return self
    ck = self._checkpointer(None if self.checkpointer is not None else path)
    if ck.path.resolve() != path.resolve(): shutil.copytree(path, ck.path, dirs_exist_ok=True)
    ck.resume = json.loads((ck.path / STATE).read_text())
    return self

## Stopping and resuming

A fit stopped part way, then resumed in a fresh learner, the way a second session would, ends where the same fit run in one go ends. The first learner trains three epochs in one go. The second gets a SIGTERM at its seventh step (`sigterm` is what the signal handler sets), saves and raises `TrainingPaused`. The third is a new learner on the same code, with other random weights to start from; it resumes from the checkpoint and finishes the fit:

In [ ]:
class At(TrainerCallback):
    "Call `fn` at the end of step `n`"
    def __init__(self, n, fn): self.n, self.fn = n, fn
    def on_step_end(self, args, state, control, **kw):
        if state.global_step == self.n: self.fn()

whole = tiny(); whole.fit_one_cycle(3, lr=1e-3)
d = Path(tempfile.mkdtemp()) / "ck"
a = tiny()
a.trainer.add_callback(At(7, lambda: setattr(a.checkpointer, "sigterm", True)))      # before the checkpointer, which then sees it at step 7
a.checkpoints(d, every=None)
test_fail(lambda: a.fit_one_cycle(3, lr=1e-3), contains="stopped (sigterm) at step 7")
st = json.loads((d / STATE).read_text())
test_eq((st["paused"], st["reason"], st["fit"], st["step"], st["fits_done"], st["trainer"]), (True, "sigterm", 1, 7, 0, "fit-1/checkpoint-7"))
sorted(p.name for p in (d / "fit-1" / "checkpoint-7").iterdir())

{'loss': '1.131', 'grad_norm': '1.131', 'learning_rate': '0', 'epoch': '0.125'}
{'loss': '1.307', 'grad_norm': '1.122', 'learning_rate': '0.0001667', 'epoch': '0.25'}
{'loss': '1.575', 'grad_norm': '0.9309', 'learning_rate': '0.0003333', 'epoch': '0.375'}
{'loss': '1.094', 'grad_norm': '0.7318', 'learning_rate': '0.0005', 'epoch': '0.5'}
{'loss': '1.096', 'grad_norm': '1.053', 'learning_rate': '0.0006667', 'epoch': '0.625'}
{'loss': '1.336', 'grad_norm': '1.126', 'learning_rate': '0.0008333', 'epoch': '0.75'}
{'loss': '0.7576', 'grad_norm': '1.058', 'learning_rate': '0.001', 'epoch': '0.875'}
{'loss': '1.439', 'grad_norm': '0.9116', 'learning_rate': '0.0009924', 'epoch': '1'}
{'eval_loss': '1.191', 'eval_accuracy': '0.4', 'eval_soft_accuracy': '0.3767', 'eval_brier': '0.26', 'eval_kl': '0.4434', 'eval_tv': '0.366', 'eval_ece': '0.07599', 'eval_score_mae': '0.838', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0.4286', 'eval_accuracy_score': '0.2', 'eval_accuracy_noul

['model.safetensors',
 'optimizer.pt',
 'rng_state.pth',
 'scheduler.pt',
 'trainer_state.json']

In [ ]:
b = tiny(seed=1).checkpoints(d, every=None).resume_from(d)
b.fit_one_cycle(3, lr=1e-3)
got, want = trained_state(b.model), trained_state(whole.model)
test_eq(set(got), set(want))
for k in want: test_close(got[k], want[k], eps=1e-5)
test_eq(len(b.recorder.metrics), 3)
test_eq([(f.get("stopped"), f.get("resumed_from_step"), f["steps"]) for f in b.fit_log.fits], [("sigterm", None, 7), (None, 7, whole.fit_log.fits[0]["steps"])])
test_eq(b.temperatures, whole.temperatures)
test_eq(json.loads((d / STATE).read_text())["fits_done"], 1)

{'loss': '1.439', 'grad_norm': '0.9116', 'learning_rate': '0.0009924', 'epoch': '1'}
{'eval_loss': '1.191', 'eval_accuracy': '0.4', 'eval_soft_accuracy': '0.3767', 'eval_brier': '0.26', 'eval_kl': '0.4434', 'eval_tv': '0.366', 'eval_ece': '0.07599', 'eval_score_mae': '0.838', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0.4286', 'eval_accuracy_score': '0.2', 'eval_accuracy_noul': '0.625', 'eval_n_unanswerable': 0, 'eval_runtime': '0.0557', 'eval_samples_per_second': '448.6', 'eval_steps_per_second': '35.89', 'epoch': '1'}
{'loss': '1.032', 'grad_norm': '1.474', 'learning_rate': '0.0009698', 'epoch': '1.125'}
{'loss': '0.66', 'grad_norm': '1.169', 'learning_rate': '0.000933', 'epoch': '1.25'}
{'loss': '0.6789', 'grad_norm': '1.537', 'learning_rate': '0.000883', 'epoch': '1.375'}
{'loss': '0.9877', 'grad_norm': '1.478', 'learning_rate': '0.0008214', 'epoch': '1.5'}
{'loss': '1.169', 'grad_norm': '1.5', 'learning_rate': '0.00075', 'epoch': '1.625'}
{'loss': '0.9796', '

The same holds with two fits, as when a head is trained and then the whole model. The run stops in the second fit; on resuming, the first is skipped (its weights come with the second's checkpoint) and the second picks up at its step:

In [ ]:
def two_fits(learn):
    learn.fit_one_cycle(1, lr=1e-3)
    learn.fit_one_cycle(2, lr=5e-4)
    return learn

whole2 = two_fits(tiny())
d2 = Path(tempfile.mkdtemp()) / "ck"
a2 = tiny()
a2.trainer.add_callback(At(4, lambda: setattr(a2.checkpointer, "sigterm", a2.fits_started == 2)))
a2.checkpoints(d2, every=None)
test_fail(lambda: two_fits(a2), contains="fit 2 stopped (sigterm) at step 4")
test_eq([p.relative_to(d2).as_posix() for p in sorted(d2.glob("fit-*/checkpoint-*"))], ["fit-2/checkpoint-4"])     # the older ones are gone
b2 = two_fits(tiny(seed=2).checkpoints(d2, every=None).resume_from(d2))
for k, v in trained_state(whole2.model).items(): test_close(trained_state(b2.model)[k], v, eps=1e-5)
test_eq([f.get("resumed_from_step") for f in b2.fit_log.fits], [None, None, 4])

{'loss': '1.131', 'grad_norm': '1.131', 'learning_rate': '0', 'epoch': '0.125'}
{'loss': '1.307', 'grad_norm': '1.122', 'learning_rate': '0.0005', 'epoch': '0.25'}
{'loss': '1.575', 'grad_norm': '0.9236', 'learning_rate': '0.001', 'epoch': '0.375'}
{'loss': '1.097', 'grad_norm': '0.7157', 'learning_rate': '0.000933', 'epoch': '0.5'}
{'loss': '1.096', 'grad_norm': '1.039', 'learning_rate': '0.00075', 'epoch': '0.625'}
{'loss': '1.34', 'grad_norm': '1.134', 'learning_rate': '0.0005', 'epoch': '0.75'}
{'loss': '0.7584', 'grad_norm': '1.071', 'learning_rate': '0.00025', 'epoch': '0.875'}
{'loss': '1.439', 'grad_norm': '0.9052', 'learning_rate': '6.699e-05', 'epoch': '1'}
{'eval_loss': '1.191', 'eval_accuracy': '0.4', 'eval_soft_accuracy': '0.3767', 'eval_brier': '0.26', 'eval_kl': '0.4434', 'eval_tv': '0.366', 'eval_ece': '0.076', 'eval_score_mae': '0.838', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0.4286', 'eval_accuracy_score': '0.3', 'eval_accuracy_noul': '0.5', '

A fit that ran out of time before the deadline stops at its first step. Here the deadline has passed already, so it stops at once, with a checkpoint of that one step:

In [ ]:
d3 = Path(tempfile.mkdtemp()) / "ck"
c = tiny().checkpoints(d3, every=None).stop_at(time.time() - 1, wrap_up=0)
test_fail(lambda: c.fit(2, lr=1e-3), contains="stopped (deadline) at step 1")
test_eq(json.loads((d3 / STATE).read_text())["reason"], "deadline")

{'loss': '1.131', 'grad_norm': '1.131', 'learning_rate': '0', 'epoch': '0.125'}
{'train_runtime': '0.0704', 'train_samples_per_second': '1704', 'train_steps_per_second': '227.1', 'train_loss': '1.131', 'epoch': '0.125'}


With `push`, the state is uploaded again as the session ends, so the Hub copy holds the paused fit's own entry, as the folder does (the upload here is a stand-in that notes what it would send):

In [ ]:
d5, sent = Path(tempfile.mkdtemp()) / "ck", []
p = tiny()
p.trainer.add_callback(At(3, lambda: setattr(p.checkpointer, "sigterm", True)))
p.checkpoints(d5, every=None, push="me/run-ckpt")
p.checkpointer._push = lambda message: sent.append((message, json.loads((d5 / STATE).read_text()))) or True
test_fail(lambda: p.fit(2, lr=1e-3), contains="stopped (sigterm) at step 3")
test_eq([(m, s["paused"], [f.get("stopped") for f in s["fits"]]) for m, s in sent], [("step 3", True, []), ("paused", True, ["sigterm"])])

{'loss': '1.131', 'grad_norm': '1.131', 'learning_rate': '0', 'epoch': '0.125'}
{'loss': '1.307', 'grad_norm': '1.122', 'learning_rate': '0.0005', 'epoch': '0.25'}
{'loss': '1.575', 'grad_norm': '0.9236', 'learning_rate': '0.001', 'epoch': '0.375'}
{'train_runtime': '0.1822', 'train_samples_per_second': '658.8', 'train_steps_per_second': '87.83', 'train_loss': '1.338', 'epoch': '0.375'}


`lr_find`'s range test runs the Trainer too, but is no fit: with checkpoints due at every step and the deadline passed, it still runs all its steps and saves nothing:

In [ ]:
d4 = Path(tempfile.mkdtemp()) / "ck"
lf = tiny().checkpoints(d4, every=0).stop_at(time.time() - 1, wrap_up=0)
lf.lr_find(steps=5)
test_eq((len(lf.lr_curve[0]), d4.exists()), (5, False))

{'loss': '1.131', 'grad_norm': '1.131', 'learning_rate': '1e-07', 'epoch': '0.125'}
{'loss': '1.307', 'grad_norm': '1.122', 'learning_rate': '5.623e-06', 'epoch': '0.25'}
{'loss': '1.575', 'grad_norm': '0.9327', 'learning_rate': '0.0003162', 'epoch': '0.375'}
{'loss': '1.094', 'grad_norm': '0.7374', 'learning_rate': '0.01778', 'epoch': '0.5'}
{'loss': '1.08', 'grad_norm': '1.479', 'learning_rate': '1', 'epoch': '0.625'}
{'train_runtime': '0.288', 'train_samples_per_second': '138.9', 'train_steps_per_second': '17.36', 'train_loss': '1.237', 'epoch': '0.625'}


After a resumed run the [training record](05a_record.ipynb) tells the whole story: each session's part of a fit is an entry of its own, the paused one with why it stopped and the resumed one with the step it resumed from, and the history runs from the first step:

In [ ]:
rec = training_record(b)["training"]
test_eq(len(rec["history"]["loss"]), len(whole.recorder.losses))
[{k: f.get(k) for k in ("steps", "stopped", "resumed_from_step")} for f in rec["fits"]]

[{'steps': 7, 'stopped': 'sigterm', 'resumed_from_step': None},
 {'steps': 24, 'stopped': None, 'resumed_from_step': 7}]

## In a job

A [sysone job](40_cloud.ipynb#sessions) passes its settings to the learner through the environment, and `Learner` reads them when it is made: `SYSONE_CHECKPOINT` (the folder) with `SYSONE_CHECKPOINT_EVERY` (minutes, 0 for epoch ends only) and `SYSONE_CHECKPOINT_PUSH` (a Hub repo), `SYSONE_DEADLINE` (a Unix time) with `SYSONE_WRAP_UP` (minutes), and `SYSONE_RESUME` (a checkpoint). Each learner the code makes gets a folder of its own, `learner-1`, `learner-2` and so on in the order they are made, which is the same in every session that runs the same code; a script that compares two models resumes each from its own.

In [ ]:
#| export
_LEARNERS = {"n": 0}

def checkpoints_from_env(learn) -> Learner:
    "Set up a job's learner from the environment: checkpoints, a deadline, and the checkpoint to resume from"
    e = os.environ
    _LEARNERS["n"] += 1
    sub = f"learner-{_LEARNERS['n']}"
    if e.get("SYSONE_CHECKPOINT"):
        every = float(e.get("SYSONE_CHECKPOINT_EVERY") or 0) or None
        push = e.get("SYSONE_CHECKPOINT_PUSH")
        learn.checkpoints(Path(e["SYSONE_CHECKPOINT"]) / sub, every=every, push=f"{push}/{sub}" if push else None)
    if e.get("SYSONE_DEADLINE"): learn.stop_at(float(e["SYSONE_DEADLINE"]), wrap_up=float(e.get("SYSONE_WRAP_UP") or 10))
    if e.get("SYSONE_RESUME"): learn.resume_from(Path(e["SYSONE_RESUME"]) / sub)
    return learn

In [ ]:
jd = Path(tempfile.mkdtemp())
os.environ |= {"SYSONE_CHECKPOINT": str(jd), "SYSONE_CHECKPOINT_EVERY": "0", "SYSONE_DEADLINE": str(time.time() + 3600), "SYSONE_WRAP_UP": "5"}
try:
    _LEARNERS["n"] = 0
    j1, j2 = tiny(), tiny()
finally:
    for k in ("SYSONE_CHECKPOINT", "SYSONE_CHECKPOINT_EVERY", "SYSONE_DEADLINE", "SYSONE_WRAP_UP"): del os.environ[k]
test_eq((j1.checkpointer.path, j2.checkpointer.path, j1.checkpointer.every, j1.checkpointer.wrap_up), (jd / "learner-1", jd / "learner-2", None, 5.0))

::: {.callout-note title="What a resume cannot redo"}
A resumed fit picks up the Trainer's state exactly, but not what the code did outside the fits. Work before the first fit runs again in every session: loading the data, building the model, and filling a [feature cache](07_cache.ipynb) (a Kaggle job carries its cache over to the next session, see [kaggle](46_kaggle.ipynb#sessions)). Code between two fits runs again too, on the model as it then is; a skipped fit's weights arrive with the fit the checkpoint belongs to. And a fit called with other arguments resumes all the same, with a warning, on a schedule that was planned for the old ones.
:::

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()